# YOLO26 Multi-Dataset Military Detection

Об'єднання 4 датасетів у 7 класів + тренування YOLO26n.

**Датасети:**
1. Military Assets Dataset (Kaggle, 12 класів)
2. AMAD-5 Aerial Military Asset Detection (Kaggle, 5 класів)
3. UAV Battle Tank Detection (Kaggle, 1 клас)
4. Aerial Person Detection (Roboflow, 10 класів)

**Підсумкові класи (7):** soldier, tank, truck, military_vehicle, civilian_vehicle, weapon, aircraft

---
### Орієнтовний час: ~5 годин на T4

---
## Крок 1: Монтуємо Google Drive

Всі результати збережуться на Drive — не пропадуть при відключенні.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Крок 2: Встановлюємо залежності

In [ ]:
!pip install -q ultralytics kagglehub roboflow pyyaml

---
## Крок 3: API ключ Roboflow

Потрібен для завантаження Aerial Person Detection.
Отримати: https://app.roboflow.com → Settings → API Keys (безкоштовно).

In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass("Введіть Roboflow API ключ:")

---
## Крок 4: Налаштування шляхів

In [ ]:
import os
from pathlib import Path

# Всі результати на Drive, щоб не пропали
DRIVE_DIR = Path("/content/drive/MyDrive/yolo_military")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

MERGED_DIR = DRIVE_DIR / "merged_dataset"
TRAIN_IMG_DIR = MERGED_DIR / "images" / "train"
TRAIN_LBL_DIR = MERGED_DIR / "labels" / "train"
VAL_IMG_DIR = MERGED_DIR / "images" / "val"
VAL_LBL_DIR = MERGED_DIR / "labels" / "val"

for d in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, VAL_IMG_DIR, VAL_LBL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Результати будуть збережено: {DRIVE_DIR}")

---
## Крок 5: Уніфікована схема класів

Всі об'єкти з різних датасетів мапляться на 7 класів:

In [ ]:
UNIFIED_CLASSES = [
    "soldier",           # 0: всі люди (військові, цивільні, пішоходи)
    "tank",              # 1: всі танки
    "truck",             # 2: вантажівки
    "military_vehicle",  # 3: інша військова техніка (howitzer, apc, jeep)
    "civilian_vehicle",  # 4: цивільний транспорт (car, van, bus, bicycle)
    "weapon",            # 5: зброя
    "aircraft",          # 6: авіація (літаки, гелікоптери, дрони)
]

print(f"Всього класів: {len(UNIFIED_CLASSES)}")
for i, name in enumerate(UNIFIED_CLASSES):
    print(f"  {i}: {name}")

---
## Крок 6: Завантаження датасетів

In [ ]:
import kagglehub

print("=" * 60)
print("Завантаження датасетів з Kaggle...")
print("=" * 60)

# 1. Military Assets Dataset (12 класів)
print("\n[1/3] Military Assets Dataset...")
military_assets_path = Path(kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
))
print(f"  -> {military_assets_path}")

# 2. AMAD-5 (5 класів)
print("\n[2/3] AMAD-5 Aerial Military Asset Detection...")
amad5_path = Path(kagglehub.dataset_download(
    "rupankarmajumdar/amad-5-aerial-military-asset-detection-dataset"
))
print(f"  -> {amad5_path}")

# 3. UAV Tank Detection (1 клас)
print("\n[3/3] UAV Battle Tank Detection...")
uav_tank_path = Path(kagglehub.dataset_download(
    "simuletic/uav-and-aerial-view-battle-tank-detection-dataset"
))
print(f"  -> {uav_tank_path}")

In [ ]:
print("=" * 60)
print("Завантаження Aerial Person Detection з Roboflow...")
print("=" * 60)

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
aerial_person_project = rf.workspace("aerial-person-detection").project("aerial-person-detection")
aerial_person_dataset = aerial_person_project.version(1).download("yolov8", location=str(DRIVE_DIR / "aerial_person_raw"))

aerial_person_path = Path(aerial_person_dataset.location)
print(f"  -> {aerial_person_path}")

---
## Крок 7: Огляд структури датасетів

Подивимось, як організований кожен датасет, і знайдемо файли з класами.

In [ ]:
def inspect_dataset(path, name):
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"Шлях: {path}")
    print(f"{'='*60}")
    
    # Показуємо перші 3 рівні структури
    for root, dirs, files in os.walk(path):
        level = Path(root).relative_to(path).parts
        if len(level) > 3:
            continue
        indent = "  " * len(level)
        print(f"{indent}{Path(root).name}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... i ще {len(files)-5} файлів")

inspect_dataset(military_assets_path, "Military Assets Dataset")
inspect_dataset(amad5_path, "AMAD-5")
inspect_dataset(uav_tank_path, "UAV Tank Detection")
inspect_dataset(aerial_person_path, "Aerial Person Detection")

---
## Крок 8: Маппінг класів

Кожна назва класу з оригінальних датасетів мапиться на наш unified ID.

In [ ]:
# Маппінг назва класу -> unified ID
CLASS_NAME_MAP = {
    # -> soldier (0)
    "camouflage_soldier": 0,
    "soldier": 0,
    "camouflaged_personnel": 0,
    "people": 0,
    "pedestrian": 0,
    "personnel": 0,
    "military_personnel": 0,
    "person": 0,
    "civilian": 0,
    # -> tank (1)
    "military_tank": 1,
    "tank": 1,
    # -> truck (2)
    "military_truck": 2,
    "truck": 2,
    # -> military_vehicle (3)
    "military_vehicle": 3,
    "howitzer": 3,
    "apc": 3,
    "military_jeep": 3,
    "armored_vehicle": 3,
    "artillery": 3,
    "jeep": 3,
    # -> civilian_vehicle (4)
    "car": 4,
    "van": 4,
    "bus": 4,
    "motor": 4,
    "bicycle": 4,
    "tricycle": 4,
    "awning-tricycle": 4,
    "civilian_vehicle": 4,
    "motorbike": 4,
    "motorcycle": 4,
    # -> weapon (5)
    "weapon": 5,
    # -> aircraft (6)
    "military_aircraft": 6,
    "military_helicopter": 6,
    "helicopter": 6,
    "aircraft": 6,
    "drone": 6,
    "airplane": 6,
}

print(f"Маппінг містить {len(CLASS_NAME_MAP)} назв класів")
print("\nКласи, які НЕ в маппінгу, будуть пропущені (напр. 'ignored regions')")

---
## Крок 9: Функція злиття датасетів

Функція читає будь-який YOLO-датасет, визначає його класи, ремапить їх і копіює в єдину структуру.

In [ ]:
import shutil
import random
from collections import defaultdict

random.seed(42)

VALID_IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def find_yolo_dataset(path):
    """
    Досліджує структуру датасету і повертає:
    - class_names: список назв класів
    - splits: dict {split_name: [(img_path, label_path)]}
    """
    path = Path(path)
    
    # Шукаємо data.yaml
    yaml_paths = list(path.rglob("data.yaml")) + list(path.rglob("dataset.yaml"))
    
    class_names = None
    if yaml_paths:
        import yaml
        with open(yaml_paths[0]) as f:
            data = yaml.safe_load(f)
        if "names" in data:
            names = data["names"]
            if isinstance(names, list):
                class_names = names
            elif isinstance(names, dict):
                class_names = [names[k] for k in sorted(names.keys())]
    
    # Шукаємо classes.txt
    if class_names is None:
        classes_txts = list(path.rglob("classes.txt"))
        if classes_txts:
            with open(classes_txts[0]) as f:
                class_names = [line.strip() for line in f if line.strip()]
    
    if class_names is None:
        # Спроба визначити за кількістю унікальних перших чисел в label файлах
        label_files = list(path.rglob("*.txt"))
        unique_ids = set()
        for lf in label_files:
            try:
                with open(lf) as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            unique_ids.add(int(line.split()[0]))
            except:
                pass
        if unique_ids:
            max_id = max(unique_ids)
            class_names = [f"class_{i}" for i in range(max_id + 1)]
    
    if class_names is None:
        raise ValueError(f"Не вдалося визначити класи для {path}")
    
    print(f"  Класи ({len(class_names)}): {class_names}")
    
    # Знаходимо всі label .txt файли (не data.yaml, не README тощо)
    label_files = []
    for lf in path.rglob("*.txt"):
        if lf.name in ("data.yaml", "dataset.yaml"):
            continue
        parent = lf.parent
        if "labels" in parent.parts or "Label" in parent.parts:
            label_files.append(lf)
    
    # Для кожного label файлу шукаємо відповідне зображення
    splits = defaultdict(list)
    
    for lf in label_files:
        stem = lf.stem
        # Шукаємо зображення в сусідніх папках
        for candidate in lf.parent.parent.rglob(f"{stem}.*"):
            if candidate.suffix.lower() in VALID_IMAGE_EXT:
                # Визначаємо split (train/val/test) за структурою папок
                rel = candidate.relative_to(path)
                parts = rel.parts
                split = "train"
                for p in parts:
                    if p.lower() in ("train", "valid", "val", "test"):
                        split = p.lower()
                        if split == "valid":
                            split = "val"
                        break
                splits[split].append((candidate, lf))
                break
    
    if not splits:
        # Fallback: шукаємо всі зображення, паруємо з label за ім'ям
        all_images = {p.stem: p for p in path.rglob("*") if p.suffix.lower() in VALID_IMAGE_EXT}
        all_labels = {p.stem: p for p in path.rglob("*.txt") if p.parent.name == "labels"}
        common = set(all_images.keys()) & set(all_labels.keys())
        if common:
            splits["train"] = [(all_images[k], all_labels[k]) for k in common]
    
    return class_names, dict(splits)


def remap_label(label_path, class_names, output_path):
    """
    Читає YOLO label файл, ремапить класи і записує новий.
    Повертає кількість об'єктів після ремаппінгу.
    """
    count = 0
    with open(label_path) as f:
        lines = f.readlines()
    
    with open(output_path, "w") as f_out:
        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            try:
                old_id = int(parts[0])
            except ValueError:
                continue
            
            if old_id >= len(class_names):
                continue
            
            old_name = class_names[old_id].lower().strip()
            
            if old_name not in CLASS_NAME_MAP:
                continue
            
            new_id = CLASS_NAME_MAP[old_name]
            new_line = f"{new_id} " + " ".join(parts[1:])
            f_out.write(new_line + "\n")
            count += 1
    
    return count


def merge_dataset(dataset_path, name, splits_to_use=None):
    """
    Зливає один датасет в загальну структуру.
    """
    print(f"\n{'='*60}")
    print(f"Обробка: {name}")
    print(f"{'='*60}")
    
    class_names, splits = find_yolo_dataset(dataset_path)
    
    print(f"  Знайдено split-ів: {list(splits.keys())}")
    
    for split_name, items in splits.items():
        if splits_to_use and split_name not in splits_to_use:
            continue
        
        target_img_dir = TRAIN_IMG_DIR if split_name == "train" else VAL_IMG_DIR
        target_lbl_dir = TRAIN_LBL_DIR if split_name == "train" else VAL_LBL_DIR
        
        kept = 0
        skipped = 0
        for img_path, lbl_path in items:
            new_name = f"{name}_{img_path.stem}"
            new_img = target_img_dir / f"{new_name}{img_path.suffix}"
            new_lbl = target_lbl_dir / f"{new_name}.txt"
            
            if new_img.exists():
                new_name = f"{name}_{img_path.stem}_{random.randint(10000, 99999)}"
                new_img = target_img_dir / f"{new_name}{img_path.suffix}"
                new_lbl = target_lbl_dir / f"{new_name}.txt"
            
            try:
                count = remap_label(lbl_path, class_names, new_lbl)
            except Exception as e:
                print(f"  Помилка при обробці {lbl_path}: {e}")
                continue
            
            if count == 0:
                new_lbl.unlink(missing_ok=True)
                skipped += 1
                continue
            
            shutil.copy2(img_path, new_img)
            kept += 1
        
        print(f"  {split_name}: {kept} зображень додано, {skipped} пропущено (немає цільових класів)")
    
    return class_names


print("✅ Функції завантажено")

---
## Крок 10: Зливаємо всі датасети

Кожен датасет обробляється окремо. Непотрібні класи (напр. "ignored regions") автоматично пропускаються.

In [ ]:
all_class_names = []

# 1. Military Assets Dataset
cn = merge_dataset(military_assets_path, "mil_assets")
all_class_names.extend(cn)

# 2. AMAD-5
cn = merge_dataset(amad5_path, "amad5")
all_class_names.extend(cn)

# 3. UAV Tank
cn = merge_dataset(uav_tank_path, "uav_tank")
all_class_names.extend(cn)

# 4. Aerial Person
cn = merge_dataset(aerial_person_path, "aerial_person")
all_class_names.extend(cn)

---
## Крок 11: Статистика об'єднаного датасету

In [ ]:
def count_images_and_objects(img_dir, lbl_dir):
    imgs = list(img_dir.glob("*"))
    total_objects = 0
    class_counts = defaultdict(int)
    
    for lbl in lbl_dir.glob("*.txt"):
        with open(lbl) as f:
            for line in f:
                line = line.strip()
                if line:
                    cls_id = int(line.split()[0])
                    class_counts[cls_id] += 1
                    total_objects += 1
    
    return len(imgs), total_objects, class_counts

train_imgs, train_objs, train_cls = count_images_and_objects(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
val_imgs, val_objs, val_cls = count_images_and_objects(VAL_IMG_DIR, VAL_LBL_DIR)

print("=" * 60)
print("Статистика об'єднаного датасету")
print("=" * 60)
print(f"\nTrain: {train_imgs} зображень, {train_objs} об'єктів")
print(f"Val:   {val_imgs} зображень, {val_objs} об'єктів")
print(f"\nРозподіл по класах (train):")
for cls_id in sorted(train_cls.keys()):
    name = UNIFIED_CLASSES[cls_id] if cls_id < len(UNIFIED_CLASSES) else f"unknown_{cls_id}"
    print(f"  {cls_id} ({name}): {train_cls[cls_id]}")
print(f"\nРозподіл по класах (val):")
for cls_id in sorted(val_cls.keys()):
    name = UNIFIED_CLASSES[cls_id] if cls_id < len(UNIFIED_CLASSES) else f"unknown_{cls_id}"
    print(f"  {cls_id} ({name}): {val_cls[cls_id]}")

---
## Крок 12: Генерація data.yaml

Створюємо файл конфігурації для YOLO.

In [ ]:
import yaml

# Абсолютний шлях на Drive
train_img_abs = str(TRAIN_IMG_DIR.resolve())
val_img_abs = str(VAL_IMG_DIR.resolve())

data_yaml = {
    "train": train_img_abs,
    "val": val_img_abs,
    "nc": len(UNIFIED_CLASSES),
    "names": UNIFIED_CLASSES,
}

yaml_path = DRIVE_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"✅ data.yaml створено: {yaml_path}")
print()
print("Вміст:")
with open(yaml_path) as f:
    print(f.read())

---
## Крок 13: Тренування YOLO26n

Використовуємо `yolo26n.pt` (найлегша модель, найшвидша).

**Моніторинг:** після кожної епохи виводиться `mAP@0.5`, `loss` тощо.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=200,
    imgsz=640,
    batch=32,
    device="cuda",
    patience=40,
    optimizer="AdamW",
    cos_lr=True,
    amp=True,
    project=str(DRIVE_DIR),
    name="military_7class",
    exist_ok=True,
    pretrained=True,
    warmup_epochs=5,
    lr0=0.001,
    augment=True,
)

---
## Крок 14: Результати тренування

Фінальні метрики на валідаційній вибірці.

In [ ]:
import pandas as pd

results_path = DRIVE_DIR / "military_7class"

if results_path.exists():
    print("Тренування завершено!")
    print(f"\nМодель: {results_path / 'weights' / 'best.pt'}")
    print(f"Чекпоінт: {results_path / 'weights' / 'last.pt'}")

    results_csv = results_path / "results.csv"
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        final = df.iloc[-1]
        print("\n=== Фінальні метрики ===")
        print(f"  mAP@0.5:      {final.get('metrics/mAP_0.5', '?'):.4f}")
        print(f"  mAP@0.5:0.95: {final.get('metrics/mAP_0.5:0.95', '?'):.4f}")
        print(f"  Precision:    {final.get('metrics/precision', '?'):.4f}")
        print(f"  Recall:       {final.get('metrics/recall', '?'):.4f}")
        print(f"  Епох:         {len(df)}")
else:
    print("Тренування ще не завершено або шлях не знайдено.")
    print(f"  Шукали: {results_path}")

---
## Крок 15: Копіювання моделі в проект

Завантажте `best.pt` на свій комп'ютер і покладіть в `detector/`.

In [ ]:
from google.colab import files

best_path = results_path / "weights" / "best.pt"

if best_path.exists():
    print(f"Завантаження {best_path}...")
    files.download(str(best_path))
    print("\nГотово!")
    print("\n--- Що робити далі ---")
    print("1. Скопіюйте best.pt в папку detector/ вашого проекту")
    print("2. Відредагуйте detector/main.py:")
    print('   model = AutoDetectionModel.from_pretrained(..., model_path="best.pt", ...)')
    print('   CLASS_FILTER = {')
    print('       "soldier": ["soldier"],')
    print('       "tank": ["tank"],')
    print('       "truck": ["truck"],')
    print('       "vehicle": ["military_vehicle", "civilian_vehicle"],')
    print('       "aircraft": ["aircraft"],')
    print('   }')
    print("3. Перезапустіть сервер і тестуйте!")
else:
    print("Файл best.pt не знайдено. Спочатку завершіть тренування.")

---
## Примітки

- Якщо після тренування якогось класу мало об'єктів (< 50), модель може погано його детектувати
- Для продовження перерваного тренування: `model = YOLO(f"{DRIVE_DIR}/military_7class/weights/last.pt")`
- Якщо CUDA Out of Memory: зменшіть `batch` до 16